# Integrated Data Engineering – Code Based Answers

Each question is written in text form followed by its corresponding code solution.

## 2.1.1 Data Format Converter

### Q1. Build a Python program that converts data between CSV, JSON, Excel, and Text formats.

In [ ]:

import pandas as pd
import json
from pathlib import Path

def convert_data(input_file, output_file):
    ext = Path(input_file).suffix.lower()
    if ext == '.csv':
        df = pd.read_csv(input_file)
    elif ext == '.json':
        with open(input_file) as f:
            data = json.load(f)
        df = pd.json_normalize(data)
    elif ext in ['.xls', '.xlsx']:
        df = pd.read_excel(input_file)
    else:
        df = pd.read_csv(input_file, sep='|')
    df.to_csv(output_file, index=False)
    return df


### Q2. How will your program handle nested JSON structures during conversion?

In [ ]:

# pandas.json_normalize() flattens nested JSON automatically
sample_json = {
    "student": {"id": 1, "name": "Alex"},
    "scores": {"math": 90, "science": 85}
}
pd.json_normalize(sample_json)


### Q3. How do you validate data types and detect missing values during conversion?

In [ ]:

def validate_data(df):
    return {
        "data_types": df.dtypes,
        "missing_values": df.isna().sum()
    }


### Q4. Design a command-line interface (CLI) for selecting input and output formats.

In [ ]:

import argparse

parser = argparse.ArgumentParser()
parser.add_argument('--input')
parser.add_argument('--output')
args = parser.parse_args()
convert_data(args.input, args.output)


### Q5. Generate a data quality report showing missing values and data types.

In [ ]:

def data_quality_report(df):
    return pd.DataFrame({
        "Data Type": df.dtypes,
        "Missing Values": df.isna().sum()
    })


## 2.2.1 Student Management System

### Q6. Design a relational database schema.

In [ ]:

-- Students(student_id, name, email)
-- Courses(course_id, course_name, credits)
-- Enrollments(student_id, course_id, grade)
-- Attendance(student_id, course_id, attendance_percent)


### Q7. Write SQL queries to calculate GPA of each student.

In [ ]:

SELECT student_id, AVG(grade) AS GPA
FROM Enrollments
GROUP BY student_id;


### Q8. Generate attendance reports.

In [ ]:

SELECT student_id, course_id, attendance_percent
FROM Attendance;


### Q9. Analyze course performance using grades.

In [ ]:

SELECT course_id, AVG(grade) AS avg_grade
FROM Enrollments
GROUP BY course_id;


### Q10. Identify at-risk students.

In [ ]:

SELECT s.student_id
FROM Students s
JOIN Enrollments e ON s.student_id = e.student_id
JOIN Attendance a ON s.student_id = a.student_id
GROUP BY s.student_id
HAVING AVG(e.grade) < 2.0 OR AVG(a.attendance_percent) < 75;


## 2.3.1 Accessing and Processing APIs

### Q11. Fetch weather data from APIs.

In [ ]:

import requests

def fetch_weather(api_url):
    response = requests.get(api_url, timeout=5)
    response.raise_for_status()
    return response.json()


### Q12. Securely manage API keys.

In [ ]:

import os
API_KEY = os.getenv('WEATHER_API_KEY')


### Q13. Handle API rate limits and failures.

In [ ]:

import time

def safe_request(url):
    for i in range(3):
        try:
            return requests.get(url).json()
        except:
            time.sleep(2 ** i)


### Q14. Normalize weather data into a common format.

In [ ]:

def normalize_weather(data):
    return {
        "temperature": data.get("temp"),
        "humidity": data.get("humidity"),
        "condition": data.get("weather")
    }


### Q15. Compare daily weather reports.

In [ ]:

def compare_weather(api1, api2):
    return api1['temperature'] - api2['temperature']


### Q16. Implement a basic weather alert system.

In [ ]:

def weather_alert(temp):
    if temp > 40:
        return "Heat Alert"
    elif temp < 0:
        return "Cold Alert"


## 2.4.1 Web Scraping

### Q17. Scrape news articles ethically.

In [ ]:

from bs4 import BeautifulSoup

def scrape_titles(url):
    html = requests.get(url, headers={'User-Agent':'Mozilla/5.0'}).text
    soup = BeautifulSoup(html, 'html.parser')
    return [h.text for h in soup.find_all('h1')]


### Q18. Ensure compliance with robots.txt.

In [ ]:

import urllib.robotparser as rp

parser = rp.RobotFileParser()
parser.set_url("https://example.com/robots.txt")
parser.read()
parser.can_fetch('*', "https://example.com")


### Q19. Extract full article details.

In [ ]:

def extract_article(soup):
    return {
        "title": soup.find('h1').text,
        "date": soup.find('time').text
    }


### Q20. Store scraped data.

In [ ]:

pd.DataFrame(data).to_csv("news.csv", index=False)


### Q21. Analyze news trends.

In [ ]:

df['category'].value_counts()


## 2.5.1 Large Dataset Handling

### Q22. Process CSV larger than memory.

In [ ]:

for chunk in pd.read_csv('large.csv', chunksize=10000):
    print(len(chunk))


### Q23. Explain chunk-based processing in pandas.

In [ ]:

# chunksize loads data in smaller blocks to reduce memory usage


### Q24. Monitor memory usage.

In [ ]:

df.memory_usage(deep=True)


### Q25. Optimize file I/O.

In [ ]:

df.to_parquet('data.parquet')


### Q26. Track progress.

In [ ]:

from tqdm import tqdm
for chunk in tqdm(pd.read_csv('large.csv', chunksize=10000)):
    pass



### Solutions Q28–Q32

**Q28 API Ingestion**
Collect weather data from API.


In [ ]:

import requests

url="https://api.open-meteo.com/v1/forecast?latitude=27.7&longitude=85.3&hourly=temperature_2m"
data=requests.get(url).json()
print("Data collected")



**Q29 Processing**
Clean and structure data.


In [ ]:

temps=data['hourly']['temperature_2m']
cleaned=[t for t in temps if t!=None]
print(len(cleaned))



**Q30 Storage**
Store data CSV


In [ ]:

import pandas as pd

df=pd.DataFrame(cleaned,columns=['Temp'])
df.to_csv('weather.csv',index=False)
print("Stored")



**Q31 API Service**
Return stored data


In [ ]:

from flask import Flask,jsonify
app=Flask(__name__)

@app.route('/data')
def getdata():
    return jsonify(cleaned)



**Q32 Visualization**
Plot temperature


In [ ]:

import matplotlib.pyplot as plt

plt.plot(cleaned)
plt.show()
